# Gaia DR4 epoch astrometry — fitting Gaia-4 b with no radial velocities

Gaia DR4 ships, for the first time, **per-CCD along-scan abscissae** — the
individual detector crossings behind each astrometric solution, rather than the
five fitted parameters. That is what lets you fit an orbit directly.

This notebook recovers the Gaia-4 b orbit from those abscissae **alone**. No RVs.

Published solution (Stefansson et al. 2024, arXiv:2410.05654, Gaia+RV):

| | |
|---|---|
| P | 571.3 d |
| e | 0.338 |
| i | 116.9 deg |
| M_c | 11.8 M_Jup |

We should land on that from astrometry only.


In [ ]:
using Nereus
using Statistics: median, quantile
using Printf


## 1. The data

`fetch_gaia_dr4_prerelease()` downloads the June 2026 pre-release VOTable into the
Nereus cache (once; afterwards it is read from disk). `read_gaia_epoch_votable`
parses one source into an `IADData` — the same structure the Hipparcos IAD reader
produces, which is why the DR4 path reuses `iad_log_likelihood` unchanged.


In [ ]:
const GAIA4_SID = 1457486023639239296

xml = get(ENV, "NEREUS_GAIA_DR4_XML", "")
if isempty(xml) || !isfile(xml)
    xml = fetch_gaia_dr4_prerelease()
end
src = read_gaia_epoch_votable(xml, GAIA4_SID)

@printf("Gaia-4: %d along-scan abscissae, G = %.2f\n", length(src.iad.t), src.g_mag)
@printf("reference position (ra0, dec0) = (%.4f, %.4f) deg\n", src.ra0, src.dec0)
@printf("baseline: %.2f yr\n", (maximum(src.iad.t) - minimum(src.iad.t)) / 365.25)


Each row is one CCD crossing: a scan angle, a parallax factor, and a measured
along-scan position with its uncertainty. The orbit shows up as a residual once
the five single-star parameters are removed.


## 2. The model

One astrometric companion, fitted in `a` (semi-major axis, AU) and `M_sec` (M_sun)
rather than in period and mass function.

**The parallax prior is not optional.** Along-scan abscissae constrain the angular
photocentre wobble a0, and a0 is proportional to M_c times parallax. Without an
independent distance the mass and the distance trade off exactly, and the posterior
runs along that degeneracy. The informative Gaia parallax pins the scale.

`inc` uses a `SinePrior` over [0, pi] so both senses of the node are reachable —
the published inclination is greater than 90 deg.


In [ ]:
const M_PRI    = 0.644     # M_sun, Stefansson+ 2024
const PLX_GAIA = 13.628    # mas
const PLX_ERR  = 0.021

target = build_target(
    M_pri = M_PRI,
    planets = (b = (
        a      = LogUniformPrior(0.3, 4.0),     # AU
        M_sec  = LogUniformPrior(0.001, 0.05),  # M_sun (11.8 M_Jup = 0.0113)
        sesinw = UniformPrior(-1.0, 1.0),
        secosw = UniformPrior(-1.0, 1.0),
        inc    = SinePrior(),
        Omega  = UniformPrior(0.0, 2pi),
        Mo     = UniformPrior(0.0, 2pi),
    ),),
    iad = src.iad,
    plx = NormalPrior(PLX_GAIA, PLX_ERR),
    M_s = M_PRI,
)

println("free parameters: ", n_unfrozen(target.params))
println(join(target.params.layout.unfrozen_names, ", "))


## 3. Fit

Parallel tempering via Pigeons. `n_rounds` doubles the sample count each round, so
cost roughly doubles too. **4 rounds is a smoke test** that finishes in a couple of
minutes; use 12 for a result you would quote.


In [ ]:
nrounds = parse(Int, get(ENV, "GAIA4_ROUNDS", "4"))   # raise to 12 for production

t0 = time()
chains, log_Z = sample_pt(target; n_rounds = nrounds, n_chains = 8,
                          seed = 42, show_report = false)
@printf("%d rounds in %.1f min,  log Z = %.2f\n", nrounds, (time()-t0)/60, log_Z)


## 4. Posterior in physical units

The sampler works in (a, M_sec, sesinw, secosw, ...); period and eccentricity are
derived. Period comes from Kepler's third law with the total mass.


In [ ]:
a_v   = vec(Array(chains[:, :a_k1, :]))
M_sec = vec(Array(chains[:, :M_sec_k1, :]))
ses   = vec(Array(chains[:, :sesinw_k1, :]))
sec   = vec(Array(chains[:, :secosw_k1, :]))
inc_v = vec(Array(chains[:, :inc_k1, :]))

e_v = ses.^2 .+ sec.^2
M_J = M_sec .* 1047.57
P_d = [365.25 * sqrt(a_v[j]^3 / (M_PRI + M_sec[j])) for j in eachindex(a_v)]

band(x) = (quantile(x,0.5), quantile(x,0.84)-quantile(x,0.5), quantile(x,0.5)-quantile(x,0.16))

println("                 median  [+1sigma, -1sigma]      published")
let (m,hi,lo)=band(P_d);            @printf("  P (d)    %8.1f [+%.1f, -%.1f]      571.3\n", m,hi,lo) end
let (m,hi,lo)=band(M_J);            @printf("  M (MJup) %8.2f [+%.2f, -%.2f]       11.8\n", m,hi,lo) end
let (m,hi,lo)=band(e_v);            @printf("  e        %8.3f [+%.3f, -%.3f]      0.338\n", m,hi,lo) end
let (m,hi,lo)=band(rad2deg.(inc_v));@printf("  i (deg)  %8.1f [+%.1f, -%.1f]      116.9\n", m,hi,lo) end


outdir = joinpath(pwd(), "gaia4_figures")
mkpath(outdir)

# Every plot function takes (chains, params, data) and writes into the
# directory given by `output` — it names the file itself.
plot_iad_residuals(chains, target.params, target.data; output = outdir)
plot_epoch_astrometry_orbit(chains, target.params, target.data; output = outdir)
plot_orbit_skyplane(chains, target.params, target.data; output = outdir)
plot_corner(chains, target.params; output = outdir)


# Nereus files model figures under a models/ subdirectory.
for (root, _, files) in walkdir(outdir), fl in files
    println("  ", relpath(joinpath(root, fl), outdir))
end


In [ ]:
outdir = joinpath(pwd(), "gaia4_figures")
mkpath(outdir)

# Every plot function takes (chains, params, data) and writes into the
# directory given by `output` — it names the file itself.
plot_iad_residuals(chains, target.params, target.data; output = outdir)
plot_epoch_astrometry_orbit(chains, target.params, target.data; output = outdir)
plot_orbit_skyplane(chains, target.params, target.data; output = outdir)
plot_corner(chains, target.params; output = outdir)


# Nereus files model figures under a models/ subdirectory.
for (root, _, files) in walkdir(outdir), fl in files
    println("  ", relpath(joinpath(root, fl), outdir))
end


save_chains(joinpath(outdir, "chains.nc"), chains, target.params;
            data = target.data, log_evidence = log_Z)
println("saved -> ", joinpath(outdir, "chains.nc"))


In [ ]:
save_chains(joinpath(outdir, "chains.nc"), chains, target.params;
            data = target.data, log_evidence = log_Z)
println("saved -> ", joinpath(outdir, "chains.nc"))


---

**Next:** `02_hd114762_joint_rv_astrometry.ipynb` adds radial velocities to the same
kind of astrometry, and shows why that combination is what measures a true mass.
